# Queueing theory - E2/E2/n queueing table using Dask

This notebook produces the typical 'average waiting time in units of service time' table ($W_q / E[ST]$) for an E2/E2/n queue, for 1 to 10 berths (servers) $n$ and berth occupancies (server utilisations) $\rho$ of 10 % to 90 %, as presented in for example the PIANC (2014) guideline. Every combination is simulated with 1,000,000 customers to approach steady-state values.

Every combination is simulated as `nr_replications` independent simulations (Dask tasks) of `nr_arr` customers, so `nr_arr * nr_replications` customers in total per table cell:
* one simulation of 100,000 customers takes about 3 s and about 350 MB of memory, so many workers can run side by side (a single simulation of 1 million customers would need several GB per worker);
* 90 cells x 10 replications = 900 tasks that Dask spreads evenly over all workers;
* the spread between the replications gives the statistical uncertainty of the table values.

Each simulation starts with an empty system; with 100,000 customers per simulation the effect of this start-up period on the averages is negligible. At high occupancies successive waiting times are strongly correlated, so these cells converge slowest.

The total computation time is roughly (number of cells) x (customers per cell) x 27 µs / (number of workers): for 90 cells with 1 million customers each about 40 minutes of CPU time, i.e. a few minutes on 12 local workers.

The calculations are distributed with [Dask](https://docs.dask.org/en/stable/deploying.html):
* locally, a `LocalCluster` starts several worker *processes* on your computer (processes rather than threads, because the simulation is pure Python and threads would be blocked by Python's global interpreter lock);
* to upscale, the same code can connect to a larger cluster (e.g. a HPC or cloud cluster) by replacing the `LocalCluster` (see section 2).

NB: Dask is an optional dependency. Install it in your environment with `pip install "dask[distributed,diagnostics]"` (or `pip install -e ".[dask]"` from the OpenQTSim folder).

In [1]:
# OpenQTSim parallel engine
import os
import openqtsim
from openqtsim.mt_engine import create_tasks, run_tasks, waiting_time_table, mmc_waiting_time_factor
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dask.distributed import Client, LocalCluster

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [ ]:
A = "E2"             # distribution of interarrival times (M: exponential, E2: erlang 2, ...)
S = "E2"             # distribution of service times (M: exponential, E2: erlang 2, ...)
servers = range(1, 11)  # numbers of berths/servers n to consider
utilisations = np.round(np.arange(0.1, 1.0, 0.1), 2)  # berth occupancy rho = lambda / (n * mu_server)
mu = 10                 # total service capacity of the system (customers per hour); each server serves mu / n
customers_per_cell = 1_000_000  # customers per combination of n and rho
nr_arr = 100_000        # customers per simulation (task)
nr_replications = customers_per_cell // nr_arr  # independent simulations per table cell
seed = 2024             # makes the whole table reproducible
n_workers = os.cpu_count()  # number of local Dask worker processes

#### 1. Create task list
Each task is one simulation: a combination of berth occupancy, number of berths and replication, with its own independent arrival and service seeds.

In [ ]:
tasks = create_tasks(A, S, servers, utilisations, nr_arr, mu=mu, nr_replications=nr_replications, seed=seed)

print(f"{len(tasks):,} simulations of {nr_arr:,} customers each ({nr_arr * nr_replications:,} customers per table cell)")
pd.DataFrame(tasks).head()

#### 2. Start a Dask cluster
The client shows a link to the Dask dashboard (open it in your browser), where you can follow the workers and the progress of the tasks while they run.

To upscale beyond your own computer, replace the `LocalCluster` by a larger cluster; the rest of the notebook stays the same. For example on a SLURM cluster such as DelftBlue (with `pip install dask-jobqueue`):

```python
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(cores=8, processes=8, memory="16GB", walltime="01:00:00")
cluster.scale(jobs=10)  # 10 jobs x 8 worker processes = 80 workers
```

or connect to a running scheduler with `client = Client("tcp://scheduler-address:8786")`.

In [ ]:
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, processes=True)
client = Client(cluster)
client

#### 3. Run the simulations in parallel

In [ ]:
%%time
results = run_tasks(tasks, client)

#### 4. Waiting time table
Average waiting time in units of average service time ($W_q / E[ST]$) for the E2/E2/n queue, averaged over the replications:

In [ ]:
def pianc_layout(df):
    """Rows: berth occupancy in %, columns: number of berths n."""
    df = df.copy()
    df.index = [f"{u:.0%}" for u in df.index]
    df.index.name = "Berth occup. (ρ/n)"
    df.columns.name = "Number of berths n"
    return df

table = waiting_time_table(tasks, results)
pianc_layout(table).style.format("{:.4f}")

Standard error of the table values (standard deviation between the replications divided by $\sqrt{\text{nr\_replications}}$):

In [ ]:
std_error = waiting_time_table(tasks, results, aggfunc="std") / np.sqrt(nr_replications)
pianc_layout(std_error).style.format("{:.4f}")

#### 5. Compare with the reference table
There is no simple closed-form solution for the E2/E2/n queue. The simulated table is compared with the table of Groenveld (2001) for 1-8 berths and 30-90 % berth occupancy. Note that the 80 % and 90 % rows of that table are given with 2 decimals only, and that the value for 4 berths at 50 % (0.0532) is larger than for 3 berths (0.0523), which is probably a typo in the source (the waiting time decreases with the number of berths).

The differences are shown as absolute values (simulated - reference), because relative differences are meaningless for the very small waiting times at low occupancies and many berths.

In [ ]:
# E2/E2/n reference table (source: Groenveld, 2001)
reference_name = "Groenveld (2001), E2/E2/n"
reference = pd.DataFrame(
    [
        [0.1310, 0.0235, 0.0062, 0.0019, 0.0007, 0.0002, 0.0001, 0.0000],
        [0.2355, 0.0576, 0.0205, 0.0085, 0.0039, 0.0019, 0.0009, 0.0005],
        [0.3904, 0.1181, 0.0523, 0.0532, 0.0142, 0.0082, 0.0050, 0.0031],
        [0.6306, 0.2222, 0.1103, 0.0639, 0.0400, 0.0265, 0.0182, 0.0128],
        [1.0391, 0.4125, 0.2275, 0.1441, 0.0988, 0.0712, 0.0532, 0.0407],
        [1.8653, 0.8300, 0.4600, 0.3300, 0.2300, 0.1900, 0.1400, 0.1200],
        [4.3590, 2.0000, 1.2000, 0.9200, 0.6500, 0.5700, 0.4400, 0.4000],
    ],
    index=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], columns=range(1, 9),
)

difference = (table - reference).reindex(index=table.index, columns=table.columns)
display(pianc_layout(reference).style.format("{:.4f}", na_rep="").set_caption(reference_name))
display(pianc_layout(difference).style.format("{:+.4f}", na_rep="").set_caption(f"Simulated - {reference_name}"))

In [ ]:
fig = go.Figure()
colors = px.colors.qualitative.Plotly
for i, n in enumerate(table.columns):
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=100 * table.index, y=table[n], error_y=dict(type="data", array=std_error[n]),
        mode="markers", marker=dict(color=color, size=8), name=f"n = {n} simulated", legendgroup=str(n),
    ))
    if n in reference.columns:
        fig.add_trace(go.Scatter(
            x=100 * reference.index, y=reference[n], mode="lines", line=dict(color=color),
            name=f"n = {n} {reference_name}", legendgroup=str(n),
        ))
fig.update_layout(
    title=f"{A}/{S}/n: average waiting time in units of service time ({nr_arr * nr_replications:,} customers per cell)",
    xaxis_title="berth occupancy ρ/n [%]", yaxis_title="W_q / E[ST] [-]", yaxis_type="log",
    height=600, template="plotly_white",
)
fig.show()

#### 6. Close the Dask cluster

In [ ]:
client.close()
cluster.close()